# Module 7: Enterprise-Grade Tooling (Type Hints & Testing)

**Concepts:** Type hints · `TypedDict` · `Optional` · **pytest** · fixtures · parametrize · mocking

**Audience:** IDP/internal tools need contracts and tests without hitting real clusters.

---

Install once: `pip install pytest`


## 1. Why type hints?

They document intent and catch bugs in CI with **mypy** or Pyright — not runtime enforcement (mostly).

Platform: API response shapes. MLOps: metric dict keys.


In [ ]:
def scale_replicas(current: int, delta: int, minimum: int = 1) -> int:
    return max(minimum, current + delta)


replicas: int = scale_replicas(3, 2)
print(replicas)


## 2. `Optional` and unions

`Optional[str]` means `str | None` — common for missing API fields.


In [ ]:
from typing import Optional


def first_label(labels: dict[str, str], key: str) -> Optional[str]:
    return labels.get(key)


print(first_label({"app": "api"}, "team"))


## 3. `TypedDict` for JSON-shaped blobs

Static checkers validate keys on dict literals assigned to TypedDict types.


In [ ]:
from typing import TypedDict


class PodStatus(TypedDict):
    name: str
    ready: bool
    restarts: int


def is_stable(p: PodStatus) -> bool:
    return p["ready"] and p["restarts"] < 5


p: PodStatus = {"name": "api-0", "ready": True, "restarts": 1}
print(is_stable(p))


## 4. Pytest basics

Tests are functions named `test_*` using plain `assert`. Run: `pytest path/to/test_file.py`.


In [ ]:
# Example test functions (run with pytest in a .py file; shown here for learning)

def is_valid_port(port: int) -> bool:
    return 1 <= port <= 65535


def test_port_ok():
    assert is_valid_port(6443)


def test_port_bad():
    assert not is_valid_port(70000)


test_port_ok()
test_port_bad()
print("inline tests passed")


## 5. Parametrize — one test, many inputs

Table-driven tests for CIDR rules, parsers, status classifiers.


In [ ]:
# pytest style (copy to test_module7.py to run with pytest)

cases = [(200, "ok"), (404, "client"), (503, "server")]


def classify(code: int) -> str:
    if 200 <= code <= 299:
        return "ok"
    if 400 <= code <= 499:
        return "client"
    if 500 <= code <= 599:
        return "server"
    return "other"


for code, expected in cases:
    assert classify(code) == expected
print("parametrize-style loop ok")


## 6. Fixtures and mocking (concept)

**Fixture** — reusable setup (fake client, temp dir).

**Mock** — replace network/database in tests (see Module 6 pattern).


In [ ]:
from unittest.mock import patch


def get_replicas(client) -> int:
    return client.fetch_replicas()


class FakeClient:
    def fetch_replicas(self) -> int:
        return 3


assert get_replicas(FakeClient()) == 3
print("fake client ok")


## 7. Capstone — typed helper + tests inline

Keep pure functions easy to test without k8s.


In [ ]:
from typing import TypedDict


class DeployPlan(TypedDict):
    env: str
    replicas: int


def allow_scale(plan: DeployPlan, max_prod: int = 10) -> bool:
    if plan["env"] == "prod":
        return plan["replicas"] <= max_prod
    return True


assert allow_scale({"env": "lab", "replicas": 99})
assert not allow_scale({"env": "prod", "replicas": 50})
print("capstone asserts ok")


---
# Exercises


**Ex 1** — Add return type `bool` to a function checking namespace name length 1..63.


In [ ]:
# TODO


**Ex 2** — TypedDict `Metric` with keys `name: str`, `value: float`; function `as_line(m) -> str`.


In [ ]:
# TODO


**Ex 3** — Write three `assert`s for `classify()` from section 5 (inline).


In [ ]:
# TODO


---
# Solutions


In [ ]:
def valid_ns(name: str) -> bool:
    return 1 <= len(name) <= 63


print(valid_ns("iim-lhp"), valid_ns(""))


In [ ]:
from typing import TypedDict


class Metric(TypedDict):
    name: str
    value: float


def as_line(m: Metric) -> str:
    return f"{m['name']}={m['value']}"


print(as_line({"name": "cpu", "value": 1.2}))


In [ ]:
def classify(code: int) -> str:
    if 200 <= code <= 299:
        return "ok"
    if 400 <= code <= 499:
        return "client"
    if 500 <= code <= 599:
        return "server"
    return "other"


assert classify(200) == "ok"
assert classify(404) == "client"
assert classify(503) == "server"
print("ok")


**Next:** Module 8 — package and ship CLI + CI.
